# 03. Detection debug

Run the SSL k-NN detector on a poisoned dataset and inspect the score distribution + per-class breakdown.

In [ ]:
import sys, pathlib, numpy as np
ROOT = pathlib.Path('..').resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.detectors.knn_detector import KNNDetector
from src.evaluation.detection_metrics import compute_detection_metrics, print_detection_report
from src.evaluation.per_class_analysis import per_class_detection_breakdown
from src.utils.data import PoisonedCIFAR, load_poisoned_dataset

POISONED = '../data/poisoned/flip_cifar10_1000.pt'
FEATURES = '../data/features/cifar10_dinov2_vits14_train.npy'

p_idx, p_lab, _ = load_poisoned_dataset(POISONED)
ds = PoisonedCIFAR('../data/raw', train=True, poison_indices=p_idx, poison_labels=p_lab)
gt = ds.get_ground_truth()
features = np.load(FEATURES)

result = KNNDetector(k=20).detect(features, gt['current_labels'])
metrics = compute_detection_metrics(result.scores, gt['is_poisoned'], gt['num_poisoned'])
print_detection_report(metrics, 'SSL k-NN (k=20)')
print(per_class_detection_breakdown(result.scores, gt['is_poisoned'], gt['current_labels'], gt['original_labels']).head(10))